In [49]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1A. Define transforms for TRAINING (Enhanced Augmentation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.RandomRotation(degrees=15),  
    
    # --- NEW AUGMENTATIONS ---
    # Shift the image by up to 10% horizontally or vertically
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), 
    # Randomly change brightness and contrast by up to 20%
    transforms.ColorJitter(brightness=0.2, contrast=0.2),     
    # -------------------------
    
    transforms.ToTensor(),         
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# 1B. Define transforms for TESTING (NO Augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),         
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# 2. Define the paths to your folders
train_dir = '../dataset/training'
test_dir = '../dataset/testing'

# 3. Load the datasets
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

# 4. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 5. Verify the data is loaded correctly
print(f"Training images: {len(train_dataset)} ({len(train_loader)} batches)")
print(f"Testing images: {len(test_dataset)} ({len(test_loader)} batches)")
print(f"Detected classes: {train_dataset.classes}")

Training images: 5600 (175 batches)
Testing images: 1600 (50 batches)
Detected classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [50]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TumorClassifierCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: 3 color channels (RGB), 224x224 image size
        
        # Block 1: Extracts basic shapes and edges
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # Shrinks image to 112x112
        
        # Block 2: Extracts more complex textures
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # Shrinks image to 56x56
        
        # Block 3: Extracts high-level structures
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2) # Shrinks image to 28x28

        # Block 4: Final visual refinement
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2) # Shrinks image to 14x14
        
        # Fully Connected (Dense) Layers for final classification
        # We have 128 channels left, and the spatial size is 14x14 (128 * 14 * 14 = 25088)
        self.fc1 = nn.Linear(128 * 14 * 14, 512)
        self.dropout = nn.Dropout(0.5) # Drops 50% of connections randomly to prevent overfitting
        self.fc2 = nn.Linear(512, 4)   # 4 output classes: glioma, meningioma, notumor, pituitary

    def forward(self, x):
        # Pass data through the conv blocks with ReLU activation and Max Pooling
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = self.pool4(F.relu(self.conv4(x)))
        
        # Flatten the 3D tensor into a 1D array for the linear layers
        x = torch.flatten(x, 1)
        
        # Pass through fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Instantiate the model and move it to the GPU to accelerate training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TumorClassifierCNN().to(device)

print(f"Model successfully built and moved to: {device}")

Model successfully built and moved to: cuda
